# 02 — Train the LoRA Adapter

Fine-tunes Qwen 2.5-7B-Instruct with QLoRA on the logistics Q&A dataset.

**Before you run:**
1. Switch Colab to a GPU runtime: Runtime → Change runtime type → T4 GPU
2. (Optional) Set `WANDB_API_KEY` in Colab secrets to enable Weights & Biases logging
3. Run `01_generate_dataset.ipynb` first so the train/val/test splits exist

**Runtime:** ~3-6 hours on a free T4 (16 GB VRAM).

In [ ]:
# Confirm GPU
!nvidia-smi

## Setup

In [ ]:
!git clone https://github.com/masonsau0/logistics-qa-lora.git
%cd logistics-qa-lora
!pip install -q -r requirements.txt

In [ ]:
# Restore dataset from Google Drive
from google.colab import drive

drive.mount("/content/drive")
!cp /content/drive/MyDrive/logistics-qa-lora/data/*.jsonl data/
!ls -lh data/*.jsonl

In [ ]:
# Optional: W&B logging
import os

from google.colab import userdata

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("W&B enabled")
    USE_WANDB = True
except Exception:
    print("WANDB_API_KEY not found in Colab secrets — training will run without W&B logging")
    USE_WANDB = False

## Train

In [ ]:
import subprocess

cmd = ["python", "-m", "src.train", "--epochs", "3", "--batch-size", "4", "--grad-accum", "4"]
if not USE_WANDB:
    cmd.append("--no-wandb")
subprocess.run(cmd, check=True)

## Save the adapter to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/logistics-qa-lora/artifacts
!cp -r artifacts/checkpoints/final /content/drive/MyDrive/logistics-qa-lora/artifacts/
!ls -lh /content/drive/MyDrive/logistics-qa-lora/artifacts/final/

## Optional: push to Hugging Face Hub

Adapters are small (~80 MB) and the Hub is a nice place to host them. You'll need an HF token with `write` scope set as `HF_TOKEN` in Colab secrets.

In [ ]:
# Uncomment to push
# from huggingface_hub import HfApi, login
# login(token=userdata.get('HF_TOKEN'))
# api = HfApi()
# api.create_repo('YOUR_HF_USERNAME/qwen25-7b-logistics-lora', exist_ok=True)
# api.upload_folder(
#     folder_path='artifacts/checkpoints/final',
#     repo_id='YOUR_HF_USERNAME/qwen25-7b-logistics-lora',
# )

### Next step
Open `03_evaluate.ipynb` to measure how much the fine-tune helped.